# 부산 폐업강도 예측 모델 — 결과 출력 전용

원본 노트북의 데이터 점검·EDA·중간 모델 비교 로그는 제외하고, **최종 LightGBM 모델의 결과만 확인**하도록 정리한 버전입니다.

실행 순서:
1. 아래 **데이터 업로드 및 계산** 셀 실행
2. 기존과 동일한 `부산_폐업모델_Master_Pack.zip` 업로드
3. **최종 결과 출력** 셀 실행

출력 항목: 최종 Test 성능 / 월별 성능 / 최신월 지역별 예측 / Feature Importance / 위험신호 순위 / 핵심 그래프

In [2]:
#@title 1. 데이터 업로드 및 최종 모델 계산
import io
import sys
import zipfile
import warnings
import subprocess
import numpy as np
import pandas as pd
from IPython.display import display, clear_output
from google.colab import files

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from lightgbm import LGBMRegressor
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'])
    from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 데이터 업로드
# ------------------------------------------------------------
uploaded = files.upload()
zip_name = next(iter(uploaded))

target_file = '부산_폐업모델_다음달예측_Ready.csv'
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name]), 'r') as z:
    if target_file not in z.namelist():
        raise FileNotFoundError(f'ZIP 안에서 {target_file} 파일을 찾지 못했습니다.')
    with z.open(target_file) as f:
        df = pd.read_csv(f, encoding='utf-8-sig')

# ------------------------------------------------------------
# 원본 최종 Feature Set
# ------------------------------------------------------------
TARGET = '다음달_폐업강도_pct_target'

numeric_features = [
    '외부방문자_YoY_pct',
    '현지인방문자_YoY_pct',
    '관광객집중도_부산방문비중_pct',
    '외부방문비중_pct',
    '체류시간_YoY_pct',
    '현지인소비_YoY_pct',
    '관광소비비중_pct',
    '외부방문자1인당관광소비_만원',
    '외부방문-관광소비_Gap_pctp',
    '관광소비_CV_3m_pct',
    '전체사업자_YoY_pct'
]

categorical_features = ['행정구', '월']
features = numeric_features + categorical_features

missing_cols = [c for c in features + [TARGET, '연월', '다음달_연월'] if c not in df.columns]
if missing_cols:
    raise KeyError(f'필수 컬럼이 없습니다: {missing_cols}')

# ------------------------------------------------------------
# 최종 Train / Test 분할 (원본 STEP 17과 동일)
# ------------------------------------------------------------
final_train = df[df['연월'] <= 202512].copy()
final_test = df[df['연월'] >= 202601].copy()

X_final_train = final_train[features]
y_final_train = final_train[TARGET]
X_final_test = final_test[features]
y_final_test = final_test[TARGET]

# ------------------------------------------------------------
# 전처리 + 최종 LightGBM
# ------------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ]
)

final_lgbm = Pipeline([
    ('preprocess', preprocessor),
    ('model', LGBMRegressor(
        objective='regression',
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        num_leaves=7,
        min_child_samples=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=2.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

final_lgbm.fit(X_final_train, y_final_train)
test_pred = final_lgbm.predict(X_final_test)

# ------------------------------------------------------------
# 평가 지표
# ------------------------------------------------------------
def metrics(y_true, pred):
    return {
        'MAE': mean_absolute_error(y_true, pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, pred)),
        'R2': r2_score(y_true, pred),
        'Bias': np.mean(pred - y_true)
    }

lgbm_metrics = metrics(y_final_test, test_pred)

dummy = DummyRegressor(strategy='mean')
dummy.fit(np.zeros((len(y_final_train), 1)), y_final_train)
dummy_pred = dummy.predict(np.zeros((len(y_final_test), 1)))
dummy_metrics = metrics(y_final_test, dummy_pred)

improvement = (dummy_metrics['MAE'] - lgbm_metrics['MAE']) / dummy_metrics['MAE'] * 100

final_result = pd.DataFrame([
    {'Model':'LightGBM_Final', 'Test_MAE':lgbm_metrics['MAE'], 'Test_RMSE':lgbm_metrics['RMSE'], 'Test_R2':lgbm_metrics['R2'], 'Bias':lgbm_metrics['Bias']},
    {'Model':'DummyMean', 'Test_MAE':dummy_metrics['MAE'], 'Test_RMSE':dummy_metrics['RMSE'], 'Test_R2':dummy_metrics['R2'], 'Bias':dummy_metrics['Bias']}
])

# ------------------------------------------------------------
# 지역별 상세 예측
# ------------------------------------------------------------
test_result = pd.DataFrame({
    'FeatureMonth': final_test['연월'].values,
    'TargetMonth': final_test['다음달_연월'].values,
    '행정구': final_test['행정구'].values,
    'Actual': y_final_test.values,
    'Predicted': test_pred
})
test_result['Error'] = test_result['Predicted'] - test_result['Actual']
test_result['AbsError'] = test_result['Error'].abs()

monthly_rows = []
for target_month, g in test_result.groupby('TargetMonth'):
    monthly_rows.append({
        'TargetMonth': target_month,
        'MAE': mean_absolute_error(g['Actual'], g['Predicted']),
        'RMSE': np.sqrt(mean_squared_error(g['Actual'], g['Predicted'])),
        'ActualMean': g['Actual'].mean(),
        'PredictedMean': g['Predicted'].mean(),
        'Bias': (g['Predicted'] - g['Actual']).mean()
    })
monthly_test = pd.DataFrame(monthly_rows)

# ------------------------------------------------------------
# Feature Importance
# ------------------------------------------------------------
fitted_preprocessor = final_lgbm.named_steps['preprocess']
fitted_model = final_lgbm.named_steps['model']
feature_names = fitted_preprocessor.get_feature_names_out()
importance = fitted_model.feature_importances_

importance_df = pd.DataFrame({'Feature':feature_names, 'Importance':importance})
importance_df['Feature_clean'] = (
    importance_df['Feature']
    .str.replace('num__', '', regex=False)
    .str.replace('cat__', '', regex=False)
)
importance_df = importance_df.sort_values('Importance', ascending=False).reset_index(drop=True)
importance_df['Importance_pct'] = importance_df['Importance'] / importance_df['Importance'].sum() * 100

# ------------------------------------------------------------
# SHAP 분석에서 원본이 확정한 위험조건을 이용한 지역 위험신호 집계
# ------------------------------------------------------------
risk_df = final_test[[
    '연월','다음달_연월','행정구',
    '외부방문비중_pct','외부방문자_YoY_pct','현지인방문자_YoY_pct',
    '전체사업자_YoY_pct','외부방문자1인당관광소비_만원','관광소비비중_pct','체류시간_YoY_pct'
]].copy()

risk_df['실제_폐업강도'] = y_final_test.values
risk_df['예측_폐업강도'] = test_pred
risk_df['절대오차'] = (risk_df['실제_폐업강도'] - risk_df['예측_폐업강도']).abs()

risk_df['R1_외부방문의존'] = risk_df['외부방문비중_pct'] >= 56.5
risk_df['R2_외부방문급증'] = risk_df['외부방문자_YoY_pct'] >= 7.7
risk_df['R3_현지인방문약화'] = risk_df['현지인방문자_YoY_pct'] < 3.5
risk_df['R4_사업자성장정체'] = risk_df['전체사업자_YoY_pct'] < 0.3
risk_df['R5_고관광객소비'] = risk_df['외부방문자1인당관광소비_만원'] >= 1.4
risk_df['R6_관광소비편중'] = risk_df['관광소비비중_pct'] >= 99.1
risk_df['R7_체류급증'] = risk_df['체류시간_YoY_pct'] >= 5.2

risk_columns = [
    'R1_외부방문의존','R2_외부방문급증','R3_현지인방문약화','R4_사업자성장정체',
    'R5_고관광객소비','R6_관광소비편중','R7_체류급증'
]
risk_df['위험신호수'] = risk_df[risk_columns].sum(axis=1)
risk_rank = risk_df.sort_values(['위험신호수','예측_폐업강도'], ascending=[False,False]).reset_index(drop=True)

signal_summary = (
    risk_df.groupby('위험신호수')
    .agg(
        관측수=('행정구','count'),
        실제폐업강도_평균=('실제_폐업강도','mean'),
        예측폐업강도_평균=('예측_폐업강도','mean'),
        실제폐업강도_중앙값=('실제_폐업강도','median')
    )
    .reset_index()
)

latest_target_month = test_result['TargetMonth'].max()
latest_prediction = (
    test_result[test_result['TargetMonth'] == latest_target_month]
    .sort_values('Predicted', ascending=False)
    .reset_index(drop=True)
)

clear_output(wait=True)
print('✅ 계산 완료 — 아래 「최종 결과 출력」 셀을 실행하세요.')

✅ 계산 완료 — 아래 「최종 결과 출력」 셀을 실행하세요.


In [3]:
#@title 2. 최종 결과 출력
import plotly.express as px
from IPython.display import display, Markdown

# 0. 발표용 핵심 수치
print('=' * 72)
print('부산 다음달 폐업강도 예측 — 최종 결과')
print('=' * 72)
print(f"최종 모델        : LightGBM")
print(f"Test MAE         : {lgbm_metrics['MAE']:.4f}")
print(f"Test RMSE        : {lgbm_metrics['RMSE']:.4f}")
print(f"Test R²          : {lgbm_metrics['R2']:.4f}")
print(f"Bias             : {lgbm_metrics['Bias']:.4f}")
print(f"Dummy 대비 개선율 : {improvement:.2f}%")
print(f"Test 기간        : {int(final_test['다음달_연월'].min())} ~ {int(final_test['다음달_연월'].max())}")

# 1. 최종 모델 성능
print('\n[1] 최종 Test 성능')
display(final_result.round(4))

# 2. 월별 성능
print('\n[2] Target Month별 성능')
display(monthly_test.round(4))

# 3. 최신월 지역별 예측
print(f'\n[3] 최신 Target Month({int(latest_target_month)}) 지역별 예측 — 예측 폐업강도 높은 순')
display(
    latest_prediction[['TargetMonth','행정구','Actual','Predicted','AbsError']]
    .rename(columns={'Actual':'실제_폐업강도','Predicted':'예측_폐업강도','AbsError':'절대오차'})
    .round(4)
)

# 4. Feature Importance
print('\n[4] LightGBM Feature Importance TOP 15')
display(
    importance_df[['Feature_clean','Importance','Importance_pct']]
    .head(15)
    .rename(columns={'Feature_clean':'Feature','Importance_pct':'Importance_pct(%)'})
    .round(2)
)

# 5. 위험신호 지역
print('\n[5] 위험신호 중첩 상위 지역')
display(
    risk_rank[['연월','다음달_연월','행정구','위험신호수','실제_폐업강도','예측_폐업강도','절대오차']]
    .head(15)
    .round(4)
)

print('\n[6] 위험신호 수에 따른 실제 폐업강도')
display(signal_summary.round(4))

# 7. 실제 vs 예측 그래프
fig = px.scatter(
    test_result,
    x='Actual', y='Predicted',
    hover_data=['행정구','TargetMonth','AbsError'],
    title='LightGBM Final Test — 실제 vs 예측 폐업강도'
)
min_val = min(test_result['Actual'].min(), test_result['Predicted'].min())
max_val = max(test_result['Actual'].max(), test_result['Predicted'].max())
fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val, line=dict(dash='dash'))
fig.update_layout(xaxis_title='실제 폐업강도 (%)', yaxis_title='예측 폐업강도 (%)', height=600)
fig.show()

# 8. 중요도 그래프
plot_df = importance_df.head(15).sort_values('Importance', ascending=True)
fig2 = px.bar(
    plot_df,
    x='Importance', y='Feature_clean', orientation='h',
    title='LightGBM Feature Importance TOP 15'
)
fig2.update_layout(xaxis_title='Importance', yaxis_title='Feature', height=600)
fig2.show()

부산 다음달 폐업강도 예측 — 최종 결과
최종 모델        : LightGBM
Test MAE         : 0.1652
Test RMSE        : 0.2145
Test R²          : 0.3422
Bias             : 0.0203
Dummy 대비 개선율 : 38.72%
Test 기간        : 202602 ~ 202605

[1] 최종 Test 성능


,Model,Test_MAE,Test_RMSE,Test_R2,Bias
0,LightGBM_Final,0.1652,0.2145,0.3422,0.0203
1,DummyMean,0.2697,0.3329,-0.5851,0.2023



[2] Target Month별 성능


,TargetMonth,MAE,RMSE,ActualMean,PredictedMean,Bias
0,202602,0.1826,0.2448,1.7529,1.9012,0.1483
1,202603,0.1417,0.1689,2.2339,2.1843,-0.0496
2,202604,0.1888,0.2501,1.9647,1.8886,-0.0761
3,202605,0.1479,0.1817,1.9158,1.9743,0.0585



[3] 최신 Target Month(202605) 지역별 예측 — 예측 폐업강도 높은 순


...

,Feature,Importance,Importance_pct(%)
0,외부방문비중_pct,154,10.64
1,월_11,139,9.60
2,외부방문-관광소비_Gap_pctp,137,9.46
3,체류시간_YoY_pct,135,9.32
4,외부방문자_YoY_pct,109,7.53
5,외부방문자1인당관광소비_만원,106,7.32
6,월_12,98,6.77
7,관광소비비중_pct,97,6.70
8,전체사업자_YoY_pct,94,6.49
9,현지인방문자_YoY_pct,84,5.80



[5] 위험신호 중첩 상위 지역


,연월,다음달_연월,행정구,위험신호수,실제_폐업강도,예측_폐업강도,절대오차
0,202602,202603,북구,6,2.1271,2.3679,0.2408
1,202602,202603,연제구,6,2.3999,2.2924,0.1076
2,202602,202603,강서구,5,2.8492,2.5419,0.3073
3,202601,202602,강서구,5,1.9575,2.3693,0.4119
4,202602,202603,사상구,5,2.7231,2.3540,0.3691
5,202603,202604,강서구,5,2.0464,2.2707,0.2243
6,202602,202603,동구,5,2.2825,2.2252,0.0573
7,202604,202605,사상구,5,2.1678,2.0867,0.0811
8,202604,202605,북구,5,1.8865,2.0742,0.1877
9,202604,202605,강서구,4,2.3644,2.2747,0.0897



[6] 위험신호 수에 따른 실제 폐업강도


,위험신호수,관측수,실제폐업강도_평균,예측폐업강도_평균,실제폐업강도_중앙값
0,0,2,1.7233,1.7511,1.7233
1,1,12,1.9031,1.8389,1.8990
2,2,13,1.8639,1.8709,1.8307
3,3,17,1.9611,1.9922,1.9351
4,4,11,1.9621,2.0758,2.0003
5,5,7,2.2733,2.2746,2.1678
6,6,2,2.2635,2.3301,2.2635
